# Historical synthetic prior plus MBDoE 2026 Lot 2

## tl;dr

This notebook measures how much the executed synthetic-must Lot 2 campaign changes calibration and practical estimability relative to the existing historical synthetic-must prior. It distinguishes observed-data information from the still-conditional CO2 flow schedule and keeps unsupported aroma directions visible rather than regularizing them into apparent identifiability.

## Context & Methods

### Key assumptions

The observed-data Fisher information matrix is computed in log-parameter coordinates,

$$F(\theta)=J(\theta)^\mathsf{T}J(\theta),\qquad
J_{ij}=\frac{\partial r_i}{\partial\log\theta_j}.$$

The prior and Lot 2 contributions are evaluated at the same combined optimum to obtain

$$F_{\mathrm{combined}}=F_{\mathrm{historical}}+F_{\mathrm{Lot2}}.$$

This decomposition separates new experimental information from changes caused only by moving the parameter estimate. The core kinetic block and reduced secondary block are re-estimated sequentially. Profile likelihood is used for selected core parameters because local FIM uncertainty alone cannot establish bounded confidence intervals.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
while ROOT.name != "pyomo-doe" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "pyomo-doe":
    raise RuntimeError("Run this notebook from inside the pyomo-doe repository")

import sys
FERMENTATION_MODEL = ROOT / "fermentation_model"
if str(FERMENTATION_MODEL) not in sys.path:
    sys.path.insert(0, str(FERMENTATION_MODEL))

from laboratory_2026 import run_estimability_historical_synthetic_plus_lot2 as analysis

analysis.run_analysis(reuse_if_current=True)
summary = json.loads((analysis.RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))
summary

## Data

In [ ]:
support = pd.read_csv(analysis.RESULTS_DIR / "lot2_measurement_support.csv")
batch_summary = pd.read_csv(analysis.RESULTS_DIR / "batch_summary.csv")
display(batch_summary.groupby("case").agg(n_batches=("batch", "nunique"), n_core_obs=("n_core_obs", "sum"), n_secondary_obs=("n_secondary_obs", "sum")))
display(support.pivot_table(index="state", values="n_observations", aggfunc="sum"))

## Results

In [ ]:
fit = pd.read_csv(analysis.RESULTS_DIR / "fit_summary.csv")
secondary_fit = pd.read_csv(analysis.RESULTS_DIR / "secondary_fit_summary.csv")
metrics = pd.read_csv(analysis.RESULTS_DIR / "fim_metrics.csv")
comparison = pd.read_csv(analysis.RESULTS_DIR / "estimability_change_historical_vs_plus_lot2.csv")
profiles = pd.read_csv(analysis.RESULTS_DIR / "profile_likelihood_summary_combined.csv")
display(fit)
display(secondary_fit)
display(metrics)
display(comparison.sort_values("std_log_ratio_combined_over_historical").head(15))
display(profiles)

In [ ]:
display(Image(filename=str(analysis.FIGURE_DIR / "estimability_historical_vs_plus_lot2.png")))
for process in ("F1", "F2", "F3"):
    display(Image(filename=str(analysis.FIGURE_DIR / "lot2_fit_overlays" / f"fit_lot2_{process}.png")))

## Takeaways

In [ ]:
display(Markdown(analysis.REPORT_PATH.read_text(encoding="utf-8")))